# Clean neighborhood (census tract) variables, pre and post

- **Paper:** (data)
- **Original notebook:** `43_update_neighborhood_clean.ipynb`
- **Reads:**
  - `data/raw/NHGIS/pre_post/nhgis0039_csv/...`
  - `data/raw/SVI/SVI_Census_tracts/...`
- **Writes:**
  - `outputs/pre_neighborhood.pkl`
  - `outputs/post_neighborhood_update.pkl`

> Note: The two saves were commented out in the original. They are turned back on here.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import zipfile




In [ ]:
# with zipfile.ZipFile(f"{DATA_DIR}/NHGIS/pre_post/nhgis0034_csv.zip","r") as zip_ref:
#     zip_ref.extractall(f"{DATA_DIR}/NHGIS/pre_post")

# read data
fi1 = pd.read_csv(f'{DATA_DIR}/NHGIS/pre_post/nhgis0039_csv/nhgis0039_ds239_20185_tract.csv',encoding='latin-1',skiprows = [1])
fi2 = pd.read_csv(f'{DATA_DIR}/NHGIS/pre_post/nhgis0039_csv/nhgis0039_ds240_20185_tract.csv',encoding='latin-1',skiprows = [1])
# fi3 = pd.read_csv(f"{DATA_DIR}/NHGIS/pre_post/nhgis0034_csv/nhgis0034_ds262_20225_tract.csv",encoding='latin-1')
# fi4 = pd.read_csv(f"{DATA_DIR}/NHGIS/pre_post/nhgis0034_csv/nhgis0034_ds263_20225_tract.csv",encoding='latin-1')
fi3 = pd.read_csv(f"{DATA_DIR}/NHGIS/pre_post/nhgis0039_csv/nhgis0039_ds267_20235_tract.csv",encoding='latin-1',skiprows = [1])
fi4 = pd.read_csv(f"{DATA_DIR}/NHGIS/pre_post/nhgis0039_csv/nhgis0039_ds268_20235_tract.csv",encoding='latin-1',skiprows = [1])
medinc = pd.read_csv(f"{DATA_DIR}/NHGIS/pre_post/nhgis0039_csv/nhgis0040_ds267_20235_tract.csv")

svi_14 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2014.csv')
svi_16 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2016_CA.csv')
svi_18 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2018_CA.csv')
svi_20 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2020.csv')
svi_22 = pd.read_csv(f'{DATA_DIR}/SVI/SVI_Census_tracts/SVI_2022.csv')


def drop_all_nan_columns(df):
    cols_to_drop = df.columns[df.iloc[1:].isna().all()]
    return df.drop(columns=cols_to_drop)


fi1 = drop_all_nan_columns(fi1)
fi2 = drop_all_nan_columns(fi2)
fi3 = drop_all_nan_columns(fi3)
fi4 = drop_all_nan_columns(fi4)

In [ ]:


fi1 = fi1[fi1['STATE']=='California']
fi1.iloc[:, 10:] = fi1.iloc[:, 10:].apply(pd.to_numeric, errors='coerce')


fi2 = fi2[fi2['STATE']=='California']
fi2.iloc[:, 10:] = fi2.iloc[:, 10:].apply(pd.to_numeric, errors='coerce')


fi3 = fi3[fi3['STATE']=='California']
fi3.iloc[:, 41:] = fi3.iloc[:, 41:].apply(pd.to_numeric, errors='coerce')

fi4 = fi4[fi4['STATE']=='California']
fi4.iloc[:, 10:] = fi4.iloc[:, 10:].apply(pd.to_numeric, errors='coerce')

fi3.columns.values

# enrolled in school, high school aged (16-19), and employed

fi1['employed_in_hs'] =  (fi1['AJYDE004']+fi1['AJYDE018'])/((fi1['AJYDE003']+fi1['AJYDE017']).replace(0, 1))


# households in the tract that are not 'limited english speaking'
fi1['english_speaking_households'] = (fi1['AJY2E002'] + fi1['AJY2E005'] +fi1[
    'AJY2E008'] + fi1['AJY2E011'] +fi1['AJY2E014'])/fi1['AJY2E001'].replace(0, 1)

# median household income
fi1['median_household_income'] = fi1['AJZAE001']

# has computer and broadband internet
fi1['has_computer_internet']  = fi1['AJ38E004']/fi1['AJ38E001'].replace(0, 1)

fi1.columns.values
df1 = fi1.loc[:,['GISJOIN', 'YEAR', 'STATE',  'COUNTY',
                 'NAME_E','employed_in_hs',"english_speaking_households",
                 "median_household_income","has_computer_internet"]].reset_index(drop=True)



# pct citizens
fi2['citizen'] = 1-((fi2['AJ68E006'])/fi2['AJ68E001'].replace(0, 1))

df2 = fi2.loc[:,["GISJOIN","citizen"]].reset_index(drop=True)



In [ ]:
# enrolled in school, high school aged (16-19), and employed
fi3.loc[:,'employed_in_hs'] = (fi3['ASPRE004']+fi3['ASPRE018'])/(fi3['ASPRE003']+fi3['ASPRE017']).replace(0, 1)

fi3.loc[:,'english_speaking_households'] = (fi3['ASQGE002'] + fi3['ASQGE005'] +fi3[
    'ASQGE008'] + fi3['ASQGE011'] +fi3['ASQGE014'])/fi3['ASQGE001'].replace(0, 1)

fi3.loc[:,'has_computer_internet']  = fi3['ASWTE004']/fi3['ASWTE001'].replace(0, 1)

# fi3.loc[:,'median_household_income'] = fi3["ASQOE001"]
medinc['median_household_income'] = medinc['ASQPE001']
medinc = medinc[['GISJOIN','median_household_income']]

medinc = medinc.loc[:,["GISJOIN","median_household_income"]].reset_index(drop=True)


df3 = fi3.loc[:,['GISJOIN', 'YEAR', 'STATE',  'COUNTY',
                 'NAME_E','employed_in_hs',"english_speaking_households",
                "has_computer_internet"]].reset_index(drop=True)




fi4['citizen'] = 1-((fi4['ASZXE006'])/fi4['ASZXE001'].replace(0, 1))
df4 =fi4.loc[:,["GISJOIN",'citizen']].reset_index(drop=True)


In [ ]:

pre = pd.merge(df1,df2, on = 'GISJOIN')

post = pd.merge(df3,df4, on = 'GISJOIN')
post = pd.merge(post,medinc, on = 'GISJOIN')

In [ ]:
# svi vars for pre
svi_14 = svi_14[["LOCATION",'FIPS','EP_UNEMP','EP_NOHSDP',
                       'EP_UNINSUR',"EP_SNGPNT",'EP_MINRTY',"EP_NOVEH"]]
svi_16 = svi_16[["LOCATION",'FIPS','E_POV','EP_UNEMP','EP_NOHSDP',
                         'EP_UNINSUR',"EP_SNGPNT",'EP_MINRTY',"EP_NOVEH"]]
svi_18 = svi_18[["LOCATION",'FIPS','E_POV','EP_UNEMP','EP_NOHSDP',
                         'EP_UNINSUR',"EP_SNGPNT",'EP_MINRTY',"EP_NOVEH"]]



svi_14.set_index('LOCATION', inplace=True)
svi_16.set_index('LOCATION', inplace=True)
svi_18.set_index('LOCATION', inplace=True)

combined_df = pd.concat([svi_14, svi_16, svi_18], axis=1, keys=['2014', '2016', '2018'])
svi_pre = combined_df.groupby(level=1, axis=1).mean()

svi_pre.reset_index(inplace=True)

all_pre = pd.merge(pre,svi_pre,left_on = 'NAME_E',right_on='LOCATION')

In [ ]:
svi_20 = svi_20[['LOCATION','FIPS','EP_POV150','EP_UNEMP','EP_HBURD','EP_NOHSDP',
                         'EP_UNINSUR',"EP_SNGPNT",'EP_MINRTY',"EP_NOVEH"]]

svi_22 = svi_22[['LOCATION',"FIPS",'EP_POV150','EP_UNEMP','EP_HBURD','EP_NOHSDP',
                         'EP_UNINSUR',"EP_SNGPNT",'EP_MINRTY',"EP_NOVEH"]]



svi_20.set_index('LOCATION', inplace=True)
svi_22.set_index('LOCATION', inplace=True)

combined_post = pd.concat([svi_20, svi_22], axis=1, keys=['2014', '2016', '2018'])
svi_post = combined_post.groupby(level=1, axis=1).mean()

svi_post.reset_index(inplace=True)
all_post = pd.merge(post,svi_post,left_on = 'NAME_E',right_on='LOCATION')

In [ ]:
all_post.drop(columns = 'LOCATION',inplace = True)

with open(f'{OUTPUT_DIR}/post_neighborhood_update.pkl','wb') as f:
  pickle.dump(all_post,f)

all_pre.drop(columns = 'LOCATION',inplace = True)

with open(f'{OUTPUT_DIR}/pre_neighborhood.pkl','wb') as f:
  pickle.dump(all_pre,f)

In [ ]:
vars_svi = all_pre

cor_mat = vars_svi.corr(numeric_only = True)
matrix = np.triu(cor_mat)

plt.figure(figsize=(12, 8))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', fmt=".2f",mask = matrix)
plt.title('Correlation Matrix')
plt.show()